<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# MarketPulse — Analytics Marketing Digital
## Notebook 1 — Contexte métier & Exploration des données
### ✅ VERSION CORRIGÉE

> **Comment lire ce corrigé :**  
> Les blocs `MÉTHODE` expliquent les choix techniques et les patterns généralisables.  
> Les blocs `INTERPRÉTATION` lisent les résultats avec les chiffres réels du dataset.  
> Les blocs `MÉTIER` font le lien entre le chiffre et la décision business.

| | |
|---|---|
| **Entreprise** | MarketPulse — agence marketing digital, Afrique de l'Ouest |
| **Période** | Juin 2022 — Mai 2024 |
| **Niveau** | Avancé |
| **Outils** | Python, DuckDB (JupySQL), matplotlib |
| **Durée estimée** | 3h à 4h |

> **Contexte :** MarketPulse gère les campagnes publicitaires de 25 clients actifs sur 4 canaux (Facebook, Google, Email, SMS) dans 6 pays d'Afrique de l'Ouest. L'objectif de ce notebook est de comprendre la structure des données, détecter les anomalies de mesure et calculer les premiers KPIs marketing avant toute préparation technique.

---
## 0. Mise en place de l'environnement

In [ ]:
!pip install jupysql==0.11.1 duckdb-engine --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import duckdb
import os, sys

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

COLORS = {
    'primary':   '#534AB7',
    'secondary': '#1D9E75',
    'warning':   '#EF9F27',
    'danger':    '#E24B4A',
    'neutral':   '#888780',
    'light':     '#EEEDFE',
}
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F9F9F8',
    'axes.grid':        True,
    'grid.alpha':       0.35,
    'font.size':        11,
})

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/projects/marketpulse/'
else:
    SAVE_PATH = './outputs/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f'📁 Environnement : {"Colab" if IN_COLAB else "Local"}')
print(f'📁 Dossier       : {SAVE_PATH}')
print('Configuration chargée ✅')

---
## 1. Connexion DuckDB et chargement des 7 tables

### MÉTHODE
`read_csv_auto()` charge les CSV directement depuis GitHub, infère les types et crée les tables en mémoire sans aucune configuration. Avantage par rapport à pandas : DuckDB permet d'écrire du SQL analytique (window functions, CTEs) directement sur ces tables sans chargement intermédiaire.

> **Ordre de chargement :** les tables de référence d'abord (`clients_agence`, `audiences`, `creatifs`), puis les tables de faits (`campagnes`, `performances_daily`, `conversions`, `contacts_crm`). Convention de lisibilité — DuckDB n'enforce pas les foreign keys.

> **Point d'attention :** `contacts_crm` contient les clients *des clients de l'agence* (ex. les abonnés d'Orange CI), pas les clients de l'agence elle-même. Ne pas confondre `client_id` (Orange CI = client de MarketPulse) avec `contact_id` (un abonné Orange CI = client final).

In [ ]:
BASE_URL = 'https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/marketpulse/data/'

conn = duckdb.connect()
conn.execute(f"""
    CREATE TABLE clients_agence    AS SELECT * FROM read_csv_auto('{BASE_URL}clients_agence.csv',    timestampformat='%Y-%m-%d');
    CREATE TABLE audiences         AS SELECT * FROM read_csv_auto('{BASE_URL}audiences.csv');
    CREATE TABLE creatifs          AS SELECT * FROM read_csv_auto('{BASE_URL}creatifs.csv',          timestampformat='%Y-%m-%d');
    CREATE TABLE campagnes         AS SELECT * FROM read_csv_auto('{BASE_URL}campagnes.csv',         timestampformat='%Y-%m-%d');
    CREATE TABLE performances_daily AS SELECT * FROM read_csv_auto('{BASE_URL}performances_daily.csv', timestampformat='%Y-%m-%d');
    CREATE TABLE conversions       AS SELECT * FROM read_csv_auto('{BASE_URL}conversions.csv',       timestampformat='%Y-%m-%d %H:%M');
    CREATE TABLE contacts_crm      AS SELECT * FROM read_csv_auto('{BASE_URL}contacts_crm.csv',      timestampformat='%Y-%m-%d');
""")

# Vérification du chargement
result = conn.execute("""
    SELECT 'clients_agence'     AS table_name, COUNT(*) AS nb_lignes,    25 AS attendu FROM clients_agence    UNION ALL
    SELECT 'audiences',                        COUNT(*),                 80            FROM audiences         UNION ALL
    SELECT 'creatifs',                         COUNT(*),                354            FROM creatifs           UNION ALL
    SELECT 'campagnes',                        COUNT(*),                354            FROM campagnes          UNION ALL
    SELECT 'performances_daily',               COUNT(*),               8861            FROM performances_daily UNION ALL
    SELECT 'conversions',                      COUNT(*),              13000            FROM conversions        UNION ALL
    SELECT 'contacts_crm',                     COUNT(*),               9500            FROM contacts_crm
""").df()
result['ecart'] = result['nb_lignes'] - result['attendu']
print(result.to_string(index=False))
print('\n✅ 7 tables chargées en mémoire DuckDB')

> **INTERPRÉTATION — Chargement des 7 tables :**
>
> | Table | Lignes | Rôle |
> |---|---|---|
> | `clients_agence` | **25** | 25 clients actifs de l'agence |
> | `audiences` | **80** | 80 segments définis sur 4 canaux |
> | `creatifs` | **354** | Un créatif par campagne (au minimum) |
> | `campagnes` | **354** | Table centrale — une ligne par campagne |
> | `performances_daily` | **8 861** | Métriques quotidiennes — environ 25 jours/campagne en moyenne |
> | `conversions` | **13 000** | Événements de conversion trackés |
> | `contacts_crm` | **9 500** | Leads et clients acquis via les campagnes |
>
> **MÉTIER :** 354 campagnes sur 24 mois = ~15 campagnes/mois. Avec 25 clients, cela représente en moyenne 0.6 campagne active par client et par mois — cohérent avec un portefeuille PME/grands comptes où certains clients lancent plusieurs campagnes simultanées et d'autres tournent sur des budgets mensuels récurrents.

In [ ]:
%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
print('%%sql prêt ✅')

---
## 2. Exploration des tables et du modèle de données

### MÉTHODE
Avant toute analyse, comprendre le schéma de données est critique. Le modèle MarketPulse est un schéma **en étoile** autour de `campagnes` :

```
clients_agence ──────────────────┐
audiences ──────────────────────┤
creatifs ────────────────────────┤──► campagnes ──► performances_daily
                                 │         │
                                 │         └──► conversions
                                 │         └──► contacts_crm
```

> **Double niveau client :** `campagnes.client_id` pointe vers `clients_agence` (Orange CI, MTN…). `contacts_crm.client_id` pointe **aussi** vers `clients_agence` mais représente les clients *finaux* d'Orange CI, pas Orange CI elle-même. Toujours qualifier les jointures par ce contexte.

In [ ]:
%%sql
-- Schéma complet : toutes les colonnes de toutes les tables
SELECT table_name, column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'main'
ORDER BY table_name, ordinal_position

In [ ]:
%%sql
-- Aperçu de la table principale
SELECT * FROM campagnes LIMIT 5

In [ ]:
%%sql
SELECT * FROM performances_daily LIMIT 5

In [ ]:
%%sql
SELECT
    client_id,
    nom,
    secteur,
    pays_principal,
    budget_mensuel_fcfa,
    type_contrat,
    objectif_principal
FROM clients_agence
ORDER BY budget_mensuel_fcfa DESC

> **INTERPRÉTATION — Structure des tables :**
>
> - `campagnes` : 14 colonnes dont `campagne_sous_performe` (variable cible ML) — ROAS final < 1.5 OU taux conversion < 1%
> - `performances_daily` : **15 colonnes** de métriques calculées (`ctr`, `roas`, `cpa_fcfa`) — **attention**, certaines sont incohérentes avec leurs sources (`clics`, `impressions`, `depenses_fcfa`). Le NB2 recalculera tout depuis les colonnes primitives
> - `clients_agence` : 25 clients couvrant 8 secteurs — les budgets vont de **500k à 50M FCFA/mois** sur un rapport de 1 à 100. Cette hétérogénéité est importante : comparer les performances en valeur absolue (dépenses) n'a pas de sens entre un client à 500k et un client à 50M. Il faut travailler en **ratios** (ROAS, CTR, CPA)
>
> **MÉTIER :** La présence de `campagne_sous_performe` dans `campagnes` signifie que la variable cible ML est déjà calculée sur la durée totale de la campagne. En NB5, on devra reconstruire cette variable à partir des données des 3 premiers jours uniquement — le challenge est de prédire le résultat final (colonne `campagne_sous_performe`) dès les premiers signaux de la campagne.

---
## 3. Diagnostic qualité — Les 6 anomalies

### MÉTHODE
Les données publicitaires multi-plateformes sont particulièrement sujettes aux anomalies car elles transitent par des APIs tierces (Facebook Ads API, Google Ads API) avant d'être centralisées. Trois types de problèmes sont courants :
1. **Incohérences calculées** — CTR qui ne correspond pas à clics/impressions (souvent un bug d'arrondi ou de division côté API)
2. **Valeurs physiquement impossibles** — clics > impressions (impossible : on ne peut pas cliquer sur une pub qu'on n'a pas vue)
3. **Erreurs de signe** — revenus négatifs (annulations de commandes mal gérées)

> **Règle fondamentale :** ne jamais corriger sans comprendre l'origine. Un ROAS de 0.00 peut être une vraie campagne de notoriété (pas de conversion attendue) ou un bug de tracking. Le contexte `objectif_campagne` permet de trancher.

In [ ]:
%%sql df_diag_perfs <<
-- DIAGNOSTIC 1 : Toutes les anomalies de performances_daily en une requête
SELECT
    COUNT(*)                                                              AS total_lignes,
    -- Clics > impressions : physiquement impossible
    SUM(CASE WHEN clics > impressions                    THEN 1 ELSE 0 END) AS clics_sup_impressions,
    -- Revenu négatif : annulation mal gérée ou bug comptable
    SUM(CASE WHEN revenu_genere_fcfa < 0                 THEN 1 ELSE 0 END) AS revenu_negatif,
    -- ROAS > 50 : aberrant sauf cas très particulier (email viral)
    SUM(CASE WHEN roas > 50                              THEN 1 ELSE 0 END) AS roas_aberrant,
    -- CTR > 20% : impossible sur Facebook/Google (plafond réel ~8%)
    SUM(CASE WHEN ctr > 0.20                             THEN 1 ELSE 0 END) AS ctr_incoherent,
    -- CTR calculé manuellement différent du CTR déclaré
    SUM(CASE WHEN impressions > 0
              AND ABS(ctr - clics * 1.0 / impressions) > 0.01
                                                         THEN 1 ELSE 0 END) AS ctr_vs_source,
    -- CPA NULL alors que conversions > 0
    SUM(CASE WHEN conversions > 0
              AND cpa_fcfa IS NULL                       THEN 1 ELSE 0 END) AS cpa_null_avec_conv
FROM performances_daily

> **INTERPRÉTATION — Anomalies dans `performances_daily` :**
>
> | Anomalie | Nb | Gravité | Traitement NB2 |
> |---|---|---|---|
> | Clics > impressions | **4** | Critique | Recalcul CTR = clics / max(impressions, clics) |
> | Revenu généré négatif | **3** | Critique | Remplacement par NULL |
> | ROAS > 50 | **2** | Critique | Recalcul depuis revenu / dépenses |
> | CTR incohérent (> 20%) | **~38** | Élevée | Recalcul systématique depuis clics/impressions |
> | CTR déclaré ≠ calculé (écart > 1pt) | **variable** | Modérée | Recalcul systématique |
>
> **MÉTIER :** Les 4 lignes avec clics > impressions proviennent d'un bug de synchronisation entre la plateforme publicitaire et le système de collecte — le clic a été enregistré dans une fenêtre temporelle différente de l'impression. C'est un problème classique dans les intégrations API temps réel. **Ces lignes biaiseraient le CTR moyen à la hausse** et donc sous-estimeraient le coût par clic réel. Elles doivent être exclues ou corrigées avant toute analyse.

In [ ]:
%%sql df_diag_camp <<
-- DIAGNOSTIC 2 : Anomalies dans la table campagnes
SELECT
    COUNT(*)                                                              AS total_campagnes,
    -- Budget dépensé > budget alloué (dépassement contractuel)
    SUM(CASE WHEN budget_depense_fcfa > budget_alloue_fcfa THEN 1 ELSE 0 END) AS depassement_budget,
    -- Campagnes sans aucune performance enregistrée
    SUM(CASE WHEN campagne_id NOT IN
              (SELECT DISTINCT campagne_id FROM performances_daily)
                                                             THEN 1 ELSE 0 END) AS sans_performance,
    -- Campagnes avec date_fin < date_debut
    SUM(CASE WHEN date_fin < date_debut                    THEN 1 ELSE 0 END) AS dates_incoherentes,
    -- Taux de campagnes sous-performantes
    ROUND(SUM(campagne_sous_performe) * 100.0 / COUNT(*), 1)              AS pct_sous_performance
FROM campagnes

> **INTERPRÉTATION — Anomalies dans `campagnes` :**
>
> - **3 dépassements de budget** : le budget dépensé dépasse le budget alloué de 5 à 25%. Cela peut indiquer une erreur de saisie du budget prévu, ou une campagne dont le cap de dépense n'a pas été correctement paramétré côté plateforme. Ces 3 campagnes seront flaggées `depassement_budget = 1` en NB2
> - **26.3% de campagnes sous-performantes** : 93 campagnes sur 354 n'ont pas atteint les seuils ROAS < 1.5 ou taux de conversion < 1%. C'est la variable cible ML du NB5
>
> **MÉTIER :** Un taux de sous-performance de 26% signifie qu'1 campagne sur 4 brûle du budget sans délivrer de résultat. Sur un budget mensuel moyen de l'agence estimé à ~30M FCFA/mois, cela représente potentiellement **7.5M FCFA/mois** gaspillés sur des campagnes en dérive. C'est l'argument économique central pour justifier le système d'alerte ML du NB5.

In [ ]:
%%sql df_diag_conv <<
-- DIAGNOSTIC 3 : Anomalies dans conversions
SELECT
    COUNT(*)                                                              AS total_conversions,
    -- Valeur NULL sur des achats (devrait toujours avoir une valeur)
    SUM(CASE WHEN type_conversion = 'Achat'
              AND valeur_fcfa IS NULL                    THEN 1 ELSE 0 END) AS achats_sans_valeur,
    -- Valeur zéro sur des achats
    SUM(CASE WHEN type_conversion = 'Achat'
              AND valeur_fcfa = 0                        THEN 1 ELSE 0 END) AS achats_valeur_zero,
    -- Délai clic-conversion aberrant (> 7 jours = douteux)
    SUM(CASE WHEN delai_clic_conversion_h > 168         THEN 1 ELSE 0 END) AS delai_aberrant,
    -- Conversions orphelines (pas de campagne correspondante)
    SUM(CASE WHEN campagne_id NOT IN
              (SELECT DISTINCT campagne_id FROM campagnes)
                                                         THEN 1 ELSE 0 END) AS orphelines,
    -- Distribution par type
    SUM(CASE WHEN type_conversion = 'Achat'           THEN 1 ELSE 0 END) AS nb_achats,
    SUM(CASE WHEN type_conversion = 'Lead'            THEN 1 ELSE 0 END) AS nb_leads,
    SUM(CASE WHEN type_conversion = 'Inscription'     THEN 1 ELSE 0 END) AS nb_inscriptions,
    SUM(CASE WHEN type_conversion = 'Appel'           THEN 1 ELSE 0 END) AS nb_appels
FROM conversions

> **INTERPRÉTATION — Anomalies dans `conversions` :**
>
> - **Achats sans valeur** : quelques conversions de type `Achat` ont une `valeur_fcfa` NULL. En NB2, elles seront imputées par la médiane des achats du même `canal_source` — on ne peut pas supprimer un achat trackable, mais on ne peut pas non plus le compter à 0 FCFA dans le calcul du ROAS
> - **Délais aberrants (> 7 jours)** : un délai entre clic et conversion > 168h est techniquement possible (attribution à fenêtre longue sur Facebook) mais suspect. Ces cas seront flaggés en NB2 sans être supprimés car ils peuvent représenter des cycles de vente longs dans le secteur B2B
> - La répartition par type de conversion révèle la nature du portefeuille : la présence de `Lead` et `Appel` confirme que l'agence travaille avec des clients dont le cycle de vente est long (banques, assurances, immobilier)
>
> **MÉTIER :** Le ROAS calculé dans `performances_daily` est une estimation basée sur l'attribution *last-click*. Les achats sans valeur et les délais longs dégradent la fiabilité de cette estimation. Un ROAS « réel » inclurait l'ensemble des conversions attribuées avec leur valeur complète — c'est ce que le NB2 construira dans `marketpulse_analytics.csv`.

In [ ]:
%%sql df_diag_recap <<
-- DIAGNOSTIC 4 : Résumé synthétique — tableau de bord qualité
SELECT 'clients_agence'     AS table_name, 25   AS attendu, COUNT(*) AS reel FROM clients_agence    UNION ALL
SELECT 'audiences',                         80,             COUNT(*) FROM audiences          UNION ALL
SELECT 'creatifs',                          354,            COUNT(*) FROM creatifs            UNION ALL
SELECT 'campagnes',                         354,            COUNT(*) FROM campagnes           UNION ALL
SELECT 'performances_daily',               8861,            COUNT(*) FROM performances_daily  UNION ALL
SELECT 'conversions',                     13000,            COUNT(*) FROM conversions         UNION ALL
SELECT 'contacts_crm',                     9500,            COUNT(*) FROM contacts_crm

---
## 4. KPI 1 — Vue globale

### MÉTHODE
On calcule tous les KPIs globaux en une seule requête pour garantir la cohérence — même périmètre de données, même instant d'exécution. **Filtre critique :** on exclut les 4 types d'anomalies identifiées en diagnostic pour obtenir des KPIs propres. `NULLIF(x, 0)` protège les divisions.

> **Définitions des KPIs :**
> - **CTR** (Click-Through Rate) = clics / impressions — mesure l'attractivité de la créa
> - **CPC** (Cost Per Click) = dépenses / clics — mesure le coût d'un visiteur
> - **CPA** (Cost Per Acquisition) = dépenses / conversions — mesure le coût d'un client
> - **ROAS** (Return on Ad Spend) = revenu / dépenses — mesure le retour sur investissement publicitaire. **ROAS = 1 signifie seuil de rentabilité**

In [ ]:
%%sql df_kpi_global <<
SELECT
    -- Volume
    COUNT(DISTINCT p.campagne_id)                                         AS nb_campagnes,
    COUNT(*)                                                              AS nb_jours_perf,
    SUM(p.impressions)                                                    AS total_impressions,
    SUM(p.clics)                                                          AS total_clics,
    SUM(p.conversions)                                                    AS total_conversions,
    -- Dépenses
    ROUND(SUM(p.depenses_fcfa) / 1e6, 2)                                  AS total_depenses_M_fcfa,
    ROUND(SUM(p.revenu_genere_fcfa) / 1e6, 2)                             AS total_revenu_M_fcfa,
    -- KPIs marketing
    ROUND(SUM(p.clics) * 100.0 / NULLIF(SUM(p.impressions), 0), 2)        AS ctr_pct,
    ROUND(SUM(p.depenses_fcfa) / NULLIF(SUM(p.clics), 0), 0)              AS cpc_fcfa,
    ROUND(SUM(p.depenses_fcfa) / NULLIF(SUM(p.conversions), 0), 0)        AS cpa_fcfa,
    ROUND(SUM(p.revenu_genere_fcfa) * 1.0 / NULLIF(SUM(p.depenses_fcfa), 0), 2) AS roas,
    -- Sous-performance
    ROUND(SUM(c.campagne_sous_performe) * 100.0 / COUNT(DISTINCT c.campagne_id), 1) AS pct_camp_sous_perf
FROM performances_daily p
JOIN campagnes c ON p.campagne_id = c.campagne_id
-- Exclusion des anomalies identifiées en diagnostic
WHERE p.clics          <= p.impressions           -- exclut les 4 lignes impossibles
  AND p.revenu_genere_fcfa >= 0                   -- exclut les 3 revenus négatifs
  AND p.roas             <= 50                    -- exclut les 2 ROAS aberrants
  AND p.ctr              <= 0.20                  -- exclut les 38 CTR incohérents

> **INTERPRÉTATION — KPIs globaux MarketPulse :**
>
> | KPI | Valeur | Lecture |
> |---|---|---|
> | Campagnes analysées | **354** | 24 mois d'activité |
> | CTR moyen | **2.86%** | 1 internaute sur 35 clique après avoir vu la pub |
> | CPC moyen | **~127 FCFA** | Prix moyen d'un visiteur sur le site client |
> | CPA moyen | **~7 287 FCFA** | Coût pour acquérir une conversion (toutes natures) |
> | ROAS moyen | **3.31×** | 1 FCFA investi rapporte 3.31 FCFA de revenu trackable |
> | Campagnes sous-perf | **26.3%** | 1 campagne sur 4 rate ses objectifs |
>
> **MÉTIER — Lecture du ROAS 3.31× :**  
> Un ROAS de 3.31 semble excellent, mais il faut le mettre en perspective. Ce ratio inclut **uniquement les revenus trackés via attribution last-click** — les ventes hors-ligne, les conversions après délai long et les effets de notoriété ne sont pas comptabilisés. En marketing digital Afrique de l'Ouest, un ROAS de 2.0 à 3.5 est généralement considéré comme satisfaisant selon le secteur. Le seuil de rentabilité réel (en intégrant les marges produit) se situe souvent entre 3.0 et 4.0 pour du e-commerce — une valeur de 3.31 est donc dans la zone de viabilité mais sans confort.
>
> **MÉTIER — Lecture du CPA 7 287 FCFA :**  
> Ce CPA agrège tous types de conversions (achats, leads, inscriptions, appels). Un lead dans le secteur bancaire vaut bien plus qu'une inscription à une newsletter. Pour avoir un CPA utile, il faut le **calculer par type de conversion et par secteur client** — ce que le NB3 fera avec des window functions.

---
## 5. KPI 2 — Performance par canal avec RANK()

### MÉTHODE
`RANK() OVER (ORDER BY roas DESC)` classe les canaux sans réduire le nombre de lignes — on peut calculer le rang dans la même requête qui calcule les KPIs. `ROUND(x * 100.0 / NULLIF(SUM(x) OVER(), 0), 1)` calcule la part de chaque canal dans le total des dépenses sans sous-requête supplémentaire.

In [ ]:
%%sql df_kpi_canal <<
WITH perf_canal AS (
    SELECT
        p.canal,
        COUNT(DISTINCT p.campagne_id)                                     AS nb_campagnes,
        SUM(p.impressions)                                                AS impressions,
        SUM(p.clics)                                                      AS clics,
        SUM(p.conversions)                                                AS conversions,
        SUM(p.depenses_fcfa)                                              AS depenses_fcfa,
        SUM(p.revenu_genere_fcfa)                                         AS revenu_fcfa,
        ROUND(SUM(p.clics) * 100.0
              / NULLIF(SUM(p.impressions), 0), 2)                        AS ctr_pct,
        ROUND(SUM(p.depenses_fcfa)
              / NULLIF(SUM(p.clics), 0), 0)                              AS cpc_fcfa,
        ROUND(SUM(p.depenses_fcfa)
              / NULLIF(SUM(p.conversions), 0), 0)                        AS cpa_fcfa,
        ROUND(SUM(p.revenu_genere_fcfa) * 1.0
              / NULLIF(SUM(p.depenses_fcfa), 0), 2)                      AS roas,
        ROUND(SUM(c.campagne_sous_performe) * 100.0
              / COUNT(DISTINCT c.campagne_id), 1)                        AS pct_sous_perf
    FROM performances_daily p
    JOIN campagnes c ON p.campagne_id = c.campagne_id
    WHERE p.clics <= p.impressions
      AND p.revenu_genere_fcfa >= 0
      AND p.roas <= 50
      AND p.ctr <= 0.20
    GROUP BY p.canal
)
SELECT
    canal,
    nb_campagnes,
    impressions,
    ctr_pct,
    cpc_fcfa,
    cpa_fcfa,
    roas,
    pct_sous_perf,
    RANK() OVER (ORDER BY roas DESC)     AS rang_roas,
    RANK() OVER (ORDER BY cpc_fcfa ASC)  AS rang_cout,
    RANK() OVER (ORDER BY pct_sous_perf ASC) AS rang_fiabilite,
    ROUND(depenses_fcfa * 100.0
          / SUM(depenses_fcfa) OVER(), 1) AS part_budget_pct
FROM perf_canal
ORDER BY rang_roas

> **INTERPRÉTATION — Classement des canaux :**
>
> | Canal | ROAS | CTR | CPC (FCFA) | Sous-perf | Rang ROAS |
> |---|---|---|---|---|---|
> | **Email** | **4.67×** | 3.89% | 46 | **0.0%** | 🥇 1 |
> | **SMS** | **3.44×** | 3.36% | 30 | 30.0% | 🥈 2 |
> | **Google** | **2.92×** | 2.77% | 226 | 33.3% | 🥉 3 |
> | **Facebook** | **2.51×** | 1.75% | 153 | 39.2% | 4 |
>
> **Lecture canal par canal :**
>
> **Email (ROAS 4.67×, CPC 46 FCFA, 0% sous-perf) :** Le canal le plus efficient du portefeuille. Le faible CPC s'explique par le coût marginal d'un email (essentiellement le coût d'envoi, pas d'enchère). Le 0% de sous-performance est remarquable — il reflète probablement le fait que les campagnes email sont envoyées sur des bases opt-in qualifiées (leads déjà chauds), ce qui garantit un taux de conversion plancher. **Recommandation :** allouer plus de budget à la constitution de bases email qualifiées.
>
> **SMS (ROAS 3.44×, CPC 30 FCFA) :** Deuxième meilleur ROAS avec le CPC le plus bas (30 FCFA). En Afrique de l'Ouest, le SMS bénéficie d'un taux d'ouverture > 95% contre ~25% pour l'email. Mais 30% de sous-performance — des campagnes SMS mal ciblées ou avec un message inadapté peuvent générer des désabonnements irréversibles.
>
> **Google (ROAS 2.92×, CPC 226 FCFA) :** Le canal le plus cher à l'acquisition mais le seul à capturer une **intention de recherche active** — l'internaute cherche déjà la solution. Un CPA élevé est acceptable si la valeur client (LTV) est suffisante. À utiliser pour les secteurs à valeur élevée (banque, assurance, immobilier).
>
> **Facebook (ROAS 2.51×, CTR 1.75%, 39.2% sous-perf) :** Le canal le plus problématique. Le faible CTR (1.75% vs 3.89% pour Email) traduit une saturation des audiences — les utilisateurs scrollent sans cliquer. Le fort taux de sous-performance (39%) confirme que beaucoup de campagnes Facebook manquent leur cible. **Ce canal nécessite le plus de surveillance et est celui qui bénéficiera le plus du système d'alerte ML.**
>
> **MÉTIER :** La hiérarchie Email > SMS > Google > Facebook est cohérente avec la théorie marketing : les canaux qui s'adressent à des audiences déjà engagées (email opt-in, SMS opt-in) ou à intention active (Google search) surperforment toujours les canaux d'interruption (Facebook display). Mais Email et SMS ont des capacités d'échelle limitées (taille des bases) — Google et Facebook restent nécessaires pour l'acquisition de nouveaux prospects.

---
## 6. KPI 3 — Performance par client agence

### MÉTHODE
On joint `performances_daily` → `campagnes` → `clients_agence` pour obtenir les KPIs par client. `RANK()` sur le ROAS permet d'identifier les clients dont les campagnes performent le mieux — et ceux qui sont en risque de départ (ROAS durablement bas).

In [ ]:
%%sql df_kpi_client <<
SELECT
    ca.client_id,
    ca.nom,
    ca.secteur,
    COUNT(DISTINCT c.campagne_id)                                         AS nb_campagnes,
    ROUND(SUM(p.depenses_fcfa) / 1e6, 2)                                  AS depenses_M_fcfa,
    ROUND(SUM(p.revenu_genere_fcfa) * 1.0
          / NULLIF(SUM(p.depenses_fcfa), 0), 2)                           AS roas,
    ROUND(SUM(p.clics) * 100.0
          / NULLIF(SUM(p.impressions), 0), 2)                             AS ctr_pct,
    ROUND(SUM(c.campagne_sous_performe) * 100.0
          / NULLIF(COUNT(DISTINCT c.campagne_id), 0), 1)                  AS pct_sous_perf,
    RANK() OVER (ORDER BY
        SUM(p.revenu_genere_fcfa) * 1.0
        / NULLIF(SUM(p.depenses_fcfa), 0) DESC)                           AS rang_roas
FROM performances_daily p
JOIN campagnes c  ON p.campagne_id = c.campagne_id
JOIN clients_agence ca ON c.client_id = ca.client_id
WHERE p.clics <= p.impressions
  AND p.revenu_genere_fcfa >= 0
  AND p.roas <= 50
  AND p.ctr  <= 0.20
GROUP BY ca.client_id, ca.nom, ca.secteur
ORDER BY rang_roas
LIMIT 15

> **INTERPRÉTATION — Performance par client :**
>
> Ce tableau révèle une **hétérogénéité forte** entre clients. Quelques constats transversaux :
>
> - Les clients du secteur **Télécoms et E-commerce** tendent à avoir les meilleurs ROAS — leurs produits ont un cycle d'achat court et un tracking simple (achat = conversion)
> - Les clients du secteur **Banque et Assurance** ont des ROAS apparemment faibles mais un CPA élevé peut être acceptable si le produit vendu (prêt, assurance) génère une LTV sur plusieurs années
> - Les clients avec `pct_sous_perf > 50%` sont en **situation critique** — plus de la moitié de leurs campagnes ratent leurs objectifs. Ce sont les candidats prioritaires pour le système d'alerte ML
>
> **MÉTIER :** Un account manager qui voit son client dans le bas du classement doit comprendre si c'est une question de **canal** (mauvais mix), de **créatif** (messages peu percutants), de **ciblage** (audience mal définie) ou de **secteur** (produit difficile à vendre en digital). Le NB3 apportera cette décomposition avec les window functions.

---
## 7. KPI 4 — Évolution mensuelle avec LAG()

### MÉTHODE
`strftime(date_perf, '%Y-%m')` extrait l'année-mois depuis une date — équivalent de `dt.to_period('M')` en pandas. `LAG(roas) OVER (ORDER BY mois)` retourne la valeur du mois précédent dans la même fenêtre. La variation `roas - LAG(roas)` mesure si la performance s'améliore ou se dégrade mois par mois.

In [ ]:
%%sql df_mensuel <<
WITH mensuel AS (
    SELECT
        strftime(p.date_perf, '%Y-%m')                                    AS mois,
        COUNT(DISTINCT p.campagne_id)                                     AS nb_campagnes_actives,
        SUM(p.depenses_fcfa)                                              AS depenses_fcfa,
        SUM(p.clics)                                                      AS clics,
        SUM(p.conversions)                                                AS conversions,
        ROUND(SUM(p.clics) * 100.0 / NULLIF(SUM(p.impressions), 0), 2)   AS ctr_pct,
        ROUND(SUM(p.revenu_genere_fcfa) * 1.0
              / NULLIF(SUM(p.depenses_fcfa), 0), 2)                      AS roas
    FROM performances_daily p
    WHERE p.clics <= p.impressions
      AND p.revenu_genere_fcfa >= 0
      AND p.roas <= 50
      AND p.ctr  <= 0.20
    GROUP BY strftime(p.date_perf, '%Y-%m')
)
SELECT
    mois,
    nb_campagnes_actives,
    depenses_fcfa,
    clics,
    conversions,
    ctr_pct,
    roas,
    LAG(roas) OVER (ORDER BY mois)                                        AS roas_mois_prec,
    ROUND(roas - LAG(roas) OVER (ORDER BY mois), 2)                      AS variation_roas,
    LAG(depenses_fcfa) OVER (ORDER BY mois)                              AS dep_mois_prec,
    ROUND(
        (depenses_fcfa - LAG(depenses_fcfa) OVER (ORDER BY mois)) * 100.0
        / NULLIF(LAG(depenses_fcfa) OVER (ORDER BY mois), 0), 1
    )                                                                     AS variation_dep_pct
FROM mensuel
ORDER BY mois

> **INTERPRÉTATION — Tendance mensuelle :**
>
> La colonne `variation_roas` révèle la **volatilité mensuelle** du portefeuille. Quelques patterns attendus :
> - Les mois de forte dépense (campagnes promotionnelles fin d'année, Ramadan) peuvent avoir un ROAS plus bas car les audiences sont saturées et les CPC augmentent
> - Les mois de faible dépense (mi-année, vacances) ont souvent un meilleur ROAS car l'agence ne dépense que sur ses campagnes les plus efficientes
>
> **MÉTIER :** Un account manager qui voit `variation_roas` fortement négatif deux mois de suite doit investiguer en priorité : est-ce un problème de **créatif** (fatigue d'audience), de **ciblage** (segment saturé), ou de **concurrence** (d'autres annonceurs enchérissent plus fort en cette période) ? Ces trois causes appellent des actions très différentes.

---
## 8. KPI 5 — Top campagnes et campagnes en dérive

### MÉTHODE
On joint `performances_daily` agrégée par `campagne_id` avec `campagnes` et `clients_agence` pour identifier les campagnes aux deux extrêmes de performance. Cette analyse bipolaire est utile pour : (1) identifier les patterns des succès et les reproduire, (2) détecter les signaux précoces des échecs et alimenter le modèle ML.

In [ ]:
%%sql df_top_camp <<
WITH camp_perf AS (
    SELECT
        p.campagne_id,
        SUM(p.depenses_fcfa)                                              AS depenses_fcfa,
        ROUND(SUM(p.revenu_genere_fcfa) * 1.0
              / NULLIF(SUM(p.depenses_fcfa), 0), 2)                      AS roas_final,
        ROUND(SUM(p.clics) * 100.0
              / NULLIF(SUM(p.impressions), 0), 2)                        AS ctr_pct,
        ROUND(SUM(p.depenses_fcfa)
              / NULLIF(SUM(p.conversions), 0), 0)                        AS cpa_fcfa,
        COUNT(DISTINCT p.date_perf)                                       AS nb_jours_actif
    FROM performances_daily p
    WHERE p.clics <= p.impressions
      AND p.revenu_genere_fcfa >= 0
      AND p.roas <= 50
      AND p.ctr  <= 0.20
    GROUP BY p.campagne_id
)
SELECT
    ca.nom                                                                AS client,
    ca.secteur,
    c.canal,
    c.objectif_campagne,
    c.pays_cible,
    cp.nb_jours_actif,
    ROUND(cp.depenses_fcfa / 1000, 0)                                     AS depenses_k_fcfa,
    cp.ctr_pct,
    cp.roas_final,
    cp.cpa_fcfa,
    c.campagne_sous_performe,
    RANK() OVER (ORDER BY cp.roas_final DESC)                             AS rang
FROM camp_perf cp
JOIN campagnes c ON cp.campagne_id = c.campagne_id
JOIN clients_agence ca ON c.client_id = ca.client_id
ORDER BY rang
LIMIT 10

In [ ]:
%%sql df_bottom_camp <<
-- Les 10 pires campagnes (sous-performantes, dépenses significatives)
WITH camp_perf AS (
    SELECT
        p.campagne_id,
        SUM(p.depenses_fcfa)                                              AS depenses_fcfa,
        ROUND(SUM(p.revenu_genere_fcfa) * 1.0
              / NULLIF(SUM(p.depenses_fcfa), 0), 2)                      AS roas_final,
        ROUND(SUM(p.clics) * 100.0
              / NULLIF(SUM(p.impressions), 0), 2)                        AS ctr_pct,
        COUNT(DISTINCT p.date_perf)                                       AS nb_jours_actif
    FROM performances_daily p
    WHERE p.clics <= p.impressions
      AND p.revenu_genere_fcfa >= 0
      AND p.roas <= 50
      AND p.ctr  <= 0.20
    GROUP BY p.campagne_id
)
SELECT
    ca.nom                                                                AS client,
    c.canal,
    c.objectif_campagne,
    cp.nb_jours_actif,
    ROUND(cp.depenses_fcfa / 1000, 0)                                     AS depenses_k_fcfa,
    cp.ctr_pct,
    cp.roas_final,
    c.campagne_sous_performe
FROM camp_perf cp
JOIN campagnes c ON cp.campagne_id = c.campagne_id
JOIN clients_agence ca ON c.client_id = ca.client_id
WHERE c.campagne_sous_performe = 1
ORDER BY cp.depenses_fcfa DESC
LIMIT 10

> **INTERPRÉTATION — Top et Bottom campagnes :**
>
> **Top campagnes :** Les meilleures performances se concentrent généralement sur :
> - Canal **Email** avec objectif **Conversion** ou **Retargeting** — audience déjà engagée
> - Campagnes **courtes** (7-14 jours) sur des bases fraîches — pas encore saturées
> - Clients du secteur **E-commerce** avec une offre promotionnelle claire
>
> **Bottom campagnes :** Les pires performances se concentrent sur :
> - Canal **Facebook** avec objectif **Notoriété** — pas de conversion directe attendue, ROAS faible par construction
> - Campagnes **longues** (> 30 jours) sans rafraîchissement de créatif — audience saturée après J+12
> - Clients dont l'offre est mal adaptée au digital (ex. produit nécessitant un conseiller humain)
>
> **MÉTIER :** Les campagnes de **Notoriété** ont structurellement un ROAS faible car leur objectif n'est pas la conversion immédiate. Calculer le `campagne_sous_performe` sur ces campagnes avec le même seuil ROAS que les campagnes de Conversion est une erreur métier — il faudrait des seuils différenciés par objectif. C'est une limite du modèle actuel que le NB2 devra traiter en créant une variable cible plus nuancée.

---
## 9. Visualisation 2×2

### MÉTHODE
Quatre visuels complémentaires en une seule figure : volume d'activité, efficience des canaux, distribution de la variable cible et évolution temporelle. On utilise la palette DataProjectLab et on sauvegarde en PNG pour réutilisation dans le dashboard Power BI.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('MarketPulse — Vue d\'ensemble analytique', fontsize=16,
             fontweight='bold', color=COLORS['primary'], y=1.01)

# ── Graphique 1 : ROAS et CTR par canal ──────────────────────────────────
ax1 = axes[0, 0]
ax1b = ax1.twinx()
canaux_labels = df_kpi_canal['canal'].tolist()
x_pos = range(len(canaux_labels))
roas_vals = df_kpi_canal['roas'].tolist()
ctr_vals  = df_kpi_canal['ctr_pct'].tolist()
bars = ax1.bar(x_pos, roas_vals, color=COLORS['primary'], alpha=0.75, label='ROAS')
ax1b.plot(x_pos, ctr_vals, color=COLORS['warning'], linewidth=2.5,
          marker='D', markersize=7, label='CTR %')
ax1.axhline(1.5, color=COLORS['danger'], linestyle='--', linewidth=1.2,
            label='Seuil sous-perf (1.5)')
for bar, v in zip(bars, roas_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{v:.2f}×', ha='center', fontsize=10, fontweight='bold',
             color=COLORS['primary'])
ax1.set_title('ROAS & CTR par canal', fontweight='bold')
ax1.set_ylabel('ROAS')
ax1b.set_ylabel('CTR (%)')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(canaux_labels)
ax1.legend(loc='upper right', fontsize=8)
ax1b.legend(loc='upper left', fontsize=8)

# ── Graphique 2 : Taux sous-performance par canal ─────────────────────────
ax2 = axes[0, 1]
colors_sp = [COLORS['danger'] if v >= 30 else COLORS['warning'] if v >= 20
             else COLORS['secondary'] for v in df_kpi_canal['pct_sous_perf'].tolist()]
bars2 = ax2.barh(canaux_labels, df_kpi_canal['pct_sous_perf'].tolist(), color=colors_sp)
ax2.axvline(26.3, color=COLORS['neutral'], linestyle='--', linewidth=1.5,
            label='Moy. globale (26.3%)')
for bar, v in zip(bars2, df_kpi_canal['pct_sous_perf'].tolist()):
    ax2.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f'{v:.1f}%', va='center', fontsize=10, fontweight='bold')
ax2.set_title('Taux de sous-performance par canal', fontweight='bold')
ax2.set_xlabel('% campagnes sous-performantes')
ax2.legend(fontsize=8)

# ── Graphique 3 : Distribution variable cible ─────────────────────────────
ax3 = axes[1, 0]
sous_perf_data = conn.execute("""
    SELECT campagne_sous_performe, COUNT(*) AS nb
    FROM campagnes GROUP BY campagne_sous_performe ORDER BY campagne_sous_performe
""").df()
labels_sp = ['À l\'heure (0)', 'Sous-perf (1)']
colors_dist = [COLORS['secondary'], COLORS['danger']]
bars3 = ax3.bar(labels_sp, sous_perf_data['nb'].tolist(), color=colors_dist)
for bar, (_, row) in zip(bars3, sous_perf_data.iterrows()):
    pct = row['nb'] / sous_perf_data['nb'].sum() * 100
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
             f'{int(row["nb"])} ({pct:.1f}%)', ha='center',
             fontweight='bold', fontsize=10)
ax3.set_title('Distribution variable cible\n(campagne_sous_performe)', fontweight='bold')
ax3.set_ylabel('Nb campagnes')

# ── Graphique 4 : ROAS mensuel ────────────────────────────────────────────
ax4 = axes[1, 1]
ax4b = ax4.twinx()
# Prendre les 12 derniers mois pour lisibilité
df_plot = df_mensuel.tail(12)
ax4.bar(range(len(df_plot)), df_plot['depenses_fcfa'].tolist(),
        color=COLORS['light'], label='Dépenses')
ax4b.plot(range(len(df_plot)), df_plot['roas'].tolist(),
          color=COLORS['primary'], linewidth=2.5, marker='o', markersize=5,
          label='ROAS mensuel')
ax4b.axhline(3.31, color=COLORS['neutral'], linestyle='--', linewidth=1,
             label='Moy. (3.31×)', alpha=0.7)
ax4b.axhline(1.5, color=COLORS['danger'], linestyle=':', linewidth=1,
             label='Seuil sous-perf')
ax4.set_title('Dépenses mensuelles & ROAS (12 derniers mois)', fontweight='bold')
ax4.set_ylabel('Dépenses (FCFA)')
ax4b.set_ylabel('ROAS')
ax4.set_xticks(range(len(df_plot)))
ax4.set_xticklabels(df_plot['mois'].tolist(), rotation=45, fontsize=8)
ax4b.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.savefig(f'{SAVE_PATH}marketpulse_vue_ensemble.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Sauvegardé : {SAVE_PATH}marketpulse_vue_ensemble.png')

> **INTERPRÉTATION — Vue d'ensemble :**
>
> **Graphique 1 (ROAS & CTR par canal) :** La hiérarchie est claire — Email domine sur les deux axes. Remarquer que SMS a un ROAS (3.44×) supérieur à Google (2.92×) malgré un format moins riche : la simplicité du message SMS et le taux d'ouverture quasi-total compensent la richesse des formats Google Ads.
>
> **Graphique 2 (sous-performance par canal) :** Email est le seul canal à 0% de sous-performance. Facebook (39.2%) et Google (33.3%) dépassent tous deux la moyenne globale de 26.3%. Ce graphique est l'argument le plus direct pour justifier le système d'alerte ML — **1 campagne Facebook sur 3 rate ses objectifs**.
>
> **Graphique 3 (variable cible) :** La distribution 73.7% / 26.3% est quasi-équilibrée — c'est favorable pour l'entraînement ML. Pas besoin de SMOTE. Le Random Forest du NB5 pourra apprendre les patterns des 93 campagnes sous-performantes sans sur-représentation artificielle.
>
> **Graphique 4 (dépenses & ROAS mensuel) :** Le ROAS fluctue entre ~2.0 et ~4.5 sur les 12 derniers mois. Les mois de forte dépense tendent à avoir un ROAS plus bas — signe de saturation des audiences ou de pression concurrentielle accrue en période de forte activité publicitaire (fin d'année, soldes).

---
## Bilan du Notebook 1

### Anomalies identifiées

| Anomalie | Nb | Table | Impact |
|---|---|---|---|
| Clics > impressions | **4** | `performances_daily` | CTR artificiellement élevé |
| Revenu généré négatif | **3** | `performances_daily` | ROAS sous-estimé |
| ROAS aberrant (> 50) | **2** | `performances_daily` | Moyenne ROAS fortement biaisée |
| CTR incohérent (> 20%) | **~38** | `performances_daily` | ~0.4% des lignes, CTR global biaised |
| Budget dépensé > alloué | **3** | `campagnes` | Dépassement contractuel non flaggé |
| Valeur conversion NULL (achats) | **~5** | `conversions` | Revenu sous-compté |

### KPIs de référence (après exclusion des anomalies)

| KPI | Valeur | Commentaire |
|---|---|---|
| Total campagnes | **354** | 24 mois · 25 clients · 4 canaux |
| ROAS moyen global | **3.31×** | Email 4.67× · Facebook 2.51× |
| CTR moyen global | **2.86%** | Email 3.89% · Facebook 1.75% |
| CPC moyen global | **127 FCFA** | SMS 30 FCFA · Google 226 FCFA |
| CPA moyen global | **7 287 FCFA** | Toutes conversions confondues |
| Campagnes sous-perf | **26.3%** (93/354) | Variable cible ML NB5 |
| Canal le plus efficient | **Email** | ROAS 4.67×, 0% sous-perf |
| Canal le plus risqué | **Facebook** | 39.2% sous-perf |

**Pour le Notebook 2 :** corriger les 6 types d'anomalies, recalculer CTR/ROAS depuis les sources, créer les features ML (`variation_ctr_j7`, `indice_saturation`, `taux_sous_perf_historique_canal`) et exporter `marketpulse_analytics.csv`.

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.